# Step 3-4: Analytical Buck Converter Model + Dataset Generation
Clean version - core cells only, testing/debug cells removed.

In [1]:
import numpy as np

In [2]:
def duty_cycle(vin, vout):
    return vout / vin

def inductor_ripple_current(vin, vout, L, fsw):
    D = duty_cycle(vin, vout)
    return ((vin - vout) * D) / (L * fsw)

def is_ccm(vin, vout, L, fsw, iload):
    delta_il = inductor_ripple_current(vin, vout, L, fsw)
    return iload > (delta_il / 2)

def output_voltage_ripple(vin, vout, L, C, fsw, esr=0.0):
    delta_il = inductor_ripple_current(vin, vout, L, fsw)
    return delta_il * (esr + 1.0 / (8.0 * C * fsw))

def rms_inductor_current(iload, delta_il):
    return iload * np.sqrt(1.0 + (1.0 / 12.0) * (delta_il / iload) ** 2)

def conduction_loss(iload, vin, vout, L, fsw, rds_on, dcr=0.0):
    D = duty_cycle(vin, vout)
    delta_il = inductor_ripple_current(vin, vout, L, fsw)
    irms = rms_inductor_current(iload, delta_il)
    p_mosfet = (irms ** 2) * rds_on * D
    p_inductor = (irms ** 2) * dcr
    return p_mosfet + p_inductor

def switching_loss(vin, iload, fsw, t_rise, t_fall):
    return 0.5 * vin * iload * (t_rise + t_fall) * fsw

def efficiency(vout, iload, p_cond, p_sw):
    p_out = vout * iload
    p_loss = p_cond + p_sw
    return p_out / (p_out + p_loss)

def analytical_design_point(vin, vout, L, C, fsw, iload, rds_on,
                             dcr=0.0, esr=0.0, t_rise=20e-9, t_fall=20e-9):
    D = duty_cycle(vin, vout)
    delta_il = inductor_ripple_current(vin, vout, L, fsw)
    ccm = is_ccm(vin, vout, L, fsw, iload)
    delta_vout = output_voltage_ripple(vin, vout, L, C, fsw, esr)
    p_cond = conduction_loss(iload, vin, vout, L, fsw, rds_on, dcr)
    p_sw = switching_loss(vin, iload, fsw, t_rise, t_fall)
    eta = efficiency(vout, iload, p_cond, p_sw)

    return {
        "vin": vin, "vout": vout, "L": L, "C": C, "fsw": fsw,
        "iload": iload, "rds_on": rds_on,
        "duty_cycle": D, "delta_il": delta_il, "ccm": ccm,
        "delta_vout": delta_vout, "p_cond": p_cond, "p_sw": p_sw,
        "p_total_loss": p_cond + p_sw, "efficiency": eta,
    }

## Parameter ranges (final, tuned for CCM/DCM balance)

In [3]:
param_ranges = {
    "L":      (1e-6, 47e-6),
    "C":      (10e-6, 470e-6),
    "fsw":    (100e3, 1e6),
    "iload":  (0.1, 5.0),
    "rds_on": (5e-3, 50e-3),
}

## Latin Hypercube Sampling

In [4]:
from scipy.stats import qmc

n_samples = 70000  # how many design points to generate

sampler = qmc.LatinHypercube(d=len(param_ranges), seed=42)
raw_samples = sampler.random(n=n_samples)   # values between 0 and 1
raw_samples.shape

(70000, 5)

In [5]:
param_names = list(param_ranges.keys())
lower_bounds = np.array([param_ranges[p][0] for p in param_names])
upper_bounds = np.array([param_ranges[p][1] for p in param_names])

scaled_samples = qmc.scale(raw_samples, lower_bounds, upper_bounds)
scaled_samples[:5]   # preview first 5 rows

array([[1.87633771e-05, 2.76475116e-04, 6.95480389e+05, 3.59680118e+00,
        3.85680109e-02],
       [3.97307017e-05, 2.11507855e-04, 2.10754179e+05, 3.73957103e+00,
        3.14391390e-02],
       [1.25595563e-05, 1.85299910e-04, 6.75413150e+05, 4.76054241e+00,
        3.73290007e-02],
       [9.69253639e-06, 2.69640070e-04, 9.70749179e+05, 3.90038207e+00,
        1.56118796e-02],
       [2.54623018e-05, 1.71589099e-04, 2.73970377e+05, 2.97602748e+00,
        4.03039282e-02]])

## Build the analytical dataset

In [6]:
import pandas as pd

results_list = []

for row in scaled_samples:
    L, C, fsw, iload, rds_on = row
    result = analytical_design_point(
        vin=12.0, vout=5.0,
        L=L, C=C, fsw=fsw, iload=iload, rds_on=rds_on,
        dcr=0.01, esr=0.01   # keeping these fixed for now
    )
    results_list.append(result)

df = pd.DataFrame(results_list)
df.shape

(70000, 15)

In [7]:
df.to_csv('analytical_dataset.csv', index=False)
print(f"Saved {len(df)} rows")

Saved 70000 rows
